In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List

# import sys
# import os
# # sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../..")))
# sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))


In [ ]:
BASE_DIR = Path("/home/std-25-353/DEV/robofl/proj_med_img/runs")  
MODELS = ["yolov9e"] # ["yolov9e", "yolo11x", "yolo26x"]
AUGS = [f"0{i}_aug_baseline" for i in range(1, 9)]  

# Output folder for all plots
OUTPUT_DIR = Path("visualizations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("tab10")
plt.rcParams.update({
    'font.size': 12,
    'figure.figsize': (12, 8),
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

def load_results(model: str, aug: str) -> pd.DataFrame:
    csv_path = BASE_DIR / model / aug / "results.csv"
    if not csv_path.exists():
        print(f"Warning: File not found → {csv_path}")
        return None
    df = pd.read_csv(csv_path)
    return df

def plot_training_curves(model: str, aug: str, df: pd.DataFrame):
    """Create main training progress plots."""
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(f"{model.upper()} - {aug}\nTraining & Validation Curves", fontsize=16, fontweight='bold')
    
    # 1. Box Loss
    axs[0, 0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', linewidth=2)
    axs[0, 0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', linewidth=2)
    axs[0, 0].set_title('Box Loss')
    axs[0, 0].set_xlabel('Epoch')
    axs[0, 0].set_ylabel('Loss')
    axs[0, 0].legend()
    
    # 2. Precision & Recall
    axs[0, 1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', linewidth=2)
    axs[0, 1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', linewidth=2)
    axs[0, 1].set_title('Precision & Recall')
    axs[0, 1].set_xlabel('Epoch')
    axs[0, 1].set_ylabel('Value')
    axs[0, 1].legend()
    
    # 3. mAP
    axs[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', linewidth=2)
    axs[1, 0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@50-95', linewidth=2)
    axs[1, 0].set_title('Mean Average Precision')
    axs[1, 0].set_xlabel('Epoch')
    axs[1, 0].set_ylabel('mAP')
    axs[1, 0].legend()
    
    # 4. Learning Rate
    axs[1, 1].plot(df['epoch'], df['lr/pg0'], label='lr/pg0', linewidth=2)
    axs[1, 1].plot(df['epoch'], df['lr/pg1'], label='lr/pg1', linewidth=2)
    axs[1, 1].plot(df['epoch'], df['lr/pg2'], label='lr/pg2', linewidth=2)
    axs[1, 1].set_title('Learning Rate Schedule')
    axs[1, 1].set_xlabel('Epoch')
    axs[1, 1].set_ylabel('Learning Rate')
    axs[1, 1].legend()
    
    plt.tight_layout()
    save_path = OUTPUT_DIR / f"{model}_{aug}_training_curves.png"
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"Saved: {save_path}")

def plot_comparison_across_augs(metric: str, title: str, ylabel: str):
    """Compare one metric across all augmentations for all models."""
    plt.figure(figsize=(14, 8))
    
    for model in MODELS:
        for aug in AUGS:
            df = load_results(model, aug)
            if df is not None:
                label = f"{model} - {aug}"
                plt.plot(df['epoch'], df[metric], label=label, linewidth=1.8)
    
    plt.title(f"Comparison: {title}", fontsize=15, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    save_path = OUTPUT_DIR / f"comparison_{metric.replace('/', '_')}.png"
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()
    print(f"Saved comparison: {save_path}")

def draw_plots():
    print("Starting visualization of YOLO training results...\n")
    
    # 1. Generate individual training curves for every model + augmentation
    print("Generating individual training curves...")
    for model in MODELS:
        for aug in AUGS:
            df = load_results(model, aug)
            if df is not None:
                plot_training_curves(model, aug, df)
    
    print("\nGenerating comparison plots across augmentations...")
    
    plot_comparison_across_augs('metrics/mAP50(B)', 'mAP@50 across all runs', 'mAP@50')
    plot_comparison_across_augs('metrics/mAP50-95(B)', 'mAP@50-95 across all runs', 'mAP@50-95')
    plot_comparison_across_augs('metrics/precision(B)', 'Precision across all runs', 'Precision')
    plot_comparison_across_augs('metrics/recall(B)', 'Recall across all runs', 'Recall')
    
    print(f"\n All visualizations saved to: ./{OUTPUT_DIR}/")
    print("You now have:")
    print("   • Individual curves for each model + augmentation")
    print("   • Comparison plots showing all 9 augmentations together")
    print("\nThese figures are ready for PowerPoint / LaTeX / your paper.")

if __name__ == "__main__":
    draw_plots()

Starting visualization of YOLO training results...

Generating individual training curves...

Generating comparison plots across augmentations...


/tmp/ipykernel_2576855/3496009637.py:86: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)


Saved comparison: visualizations/comparison_metrics_mAP50(B).png
Saved comparison: visualizations/comparison_metrics_mAP50-95(B).png
Saved comparison: visualizations/comparison_metrics_precision(B).png
Saved comparison: visualizations/comparison_metrics_recall(B).png

 All visualizations saved to: ./visualizations/
You now have:
   • Individual curves for each model + augmentation
   • Comparison plots showing all 9 augmentations together

These figures are ready for PowerPoint / LaTeX / your paper.


In [7]:
import sys
import os
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../..")))
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../")))


In [9]:
os.getcwd()

'/home/std-25-353/DEV/robofl/proj_med_img/src/evaluation'